# Online vs batch Pearson r — equivalence check

Shows that `ProfileCorrAccumulator` (online sufficient statistics) produces identical results to computing Pearson r in batch with `numpy.corrcoef`, across a simulated multi-interval scenario at 1 bp and 32 bp resolution.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)

## ProfileCorrAccumulator (copy from collect_predictions.py)

In [2]:
class ProfileCorrAccumulator:
    def __init__(self, n_tracks):
        z = np.zeros(n_tracks, dtype=np.float64)
        self.n      = np.zeros(n_tracks, dtype=np.int64)
        self.sum_x  = z.copy()
        self.sum_y  = z.copy()
        self.sum_xx = z.copy()
        self.sum_yy = z.copy()
        self.sum_xy = z.copy()

    def update(self, x, y):
        self.n      += x.shape[0]
        self.sum_x  += x.sum(axis=0)
        self.sum_y  += y.sum(axis=0)
        self.sum_xx += (x ** 2).sum(axis=0)
        self.sum_yy += (y ** 2).sum(axis=0)
        self.sum_xy += (x * y).sum(axis=0)

    def result(self):
        n = self.n.astype(np.float64)
        num = n * self.sum_xy - self.sum_x * self.sum_y
        den = np.sqrt(
            np.maximum(0.0, n * self.sum_xx - self.sum_x ** 2)
            * np.maximum(0.0, n * self.sum_yy - self.sum_y ** 2)
        )
        with np.errstate(invalid='ignore', divide='ignore'):
            return np.where(den > 0, num / den, np.nan)

## Dummy data

Simulate 5 test intervals of varying lengths, each with predictions and observations for 4 tracks (2 samples × 2 strands). Coverage values are log-normal to mimic RNA-seq signal.

In [3]:
N_INTERVALS = 5
N_TRACKS    = 4   # 2 samples × 2 strands
BIN_SIZE    = 32

# Each interval has a random length (multiples of 32 for clean binning)
interval_lengths = rng.integers(5, 30, size=N_INTERVALS) * BIN_SIZE
print('Interval lengths (bp):', interval_lengths)

# Generate pred and obs for each interval.
# Add correlated noise so Pearson r is meaningful (not random).
intervals = []
for length in interval_lengths:
    signal = rng.exponential(scale=2.0, size=(length, N_TRACKS))
    pred   = signal * rng.lognormal(0, 0.3, size=(length, N_TRACKS))
    obs    = signal * rng.lognormal(0, 0.3, size=(length, N_TRACKS))
    intervals.append((np.log1p(pred), np.log1p(obs)))

Interval lengths (bp): [832 640 544 352 384]


## Online accumulator (interval by interval)

In [4]:
acc_1bp  = ProfileCorrAccumulator(N_TRACKS)
acc_32bp = ProfileCorrAccumulator(N_TRACKS)

for pred_1bp, obs_1bp in intervals:
    # 1 bp
    acc_1bp.update(pred_1bp, obs_1bp)

    # 32 bp: mean-pool into bins
    n_bins   = pred_1bp.shape[0] // BIN_SIZE
    pred_32  = pred_1bp[:n_bins * BIN_SIZE].reshape(n_bins, BIN_SIZE, -1).mean(axis=1)
    obs_32   = obs_1bp [:n_bins * BIN_SIZE].reshape(n_bins, BIN_SIZE, -1).mean(axis=1)
    acc_32bp.update(pred_32, obs_32)

r_online_1bp  = acc_1bp.result()
r_online_32bp = acc_32bp.result()
print('Online r (1 bp) :', r_online_1bp)
print('Online r (32 bp):', r_online_32bp)

Online r (1 bp) : [0.91771062 0.91538167 0.91298279 0.92017669]
Online r (32 bp): [0.92557708 0.9127592  0.90837102 0.93356949]


## Batch reference (concatenate all, then numpy.corrcoef)

In [5]:
# Concatenate all intervals
all_pred_1bp = np.concatenate([p for p, _ in intervals], axis=0)
all_obs_1bp  = np.concatenate([o for _, o in intervals], axis=0)

r_batch_1bp = np.array([
    np.corrcoef(all_pred_1bp[:, t], all_obs_1bp[:, t])[0, 1]
    for t in range(N_TRACKS)
])

# 32 bp
all_pred_32bp = []
all_obs_32bp  = []
for pred_1bp, obs_1bp in intervals:
    n_bins = pred_1bp.shape[0] // BIN_SIZE
    all_pred_32bp.append(pred_1bp[:n_bins * BIN_SIZE].reshape(n_bins, BIN_SIZE, -1).mean(axis=1))
    all_obs_32bp.append (obs_1bp [:n_bins * BIN_SIZE].reshape(n_bins, BIN_SIZE, -1).mean(axis=1))
all_pred_32bp = np.concatenate(all_pred_32bp, axis=0)
all_obs_32bp  = np.concatenate(all_obs_32bp,  axis=0)

r_batch_32bp = np.array([
    np.corrcoef(all_pred_32bp[:, t], all_obs_32bp[:, t])[0, 1]
    for t in range(N_TRACKS)
])

print('Batch r (1 bp) :', r_batch_1bp)
print('Batch r (32 bp):', r_batch_32bp)

Batch r (1 bp) : [0.91771062 0.91538167 0.91298279 0.92017669]
Batch r (32 bp): [0.92557708 0.9127592  0.90837102 0.93356949]


## Equivalence check

In [6]:
pd.DataFrame({
    'track':       range(N_TRACKS),
    'online_1bp':  r_online_1bp,
    'batch_1bp':   r_batch_1bp,
    'diff_1bp':    np.abs(r_online_1bp - r_batch_1bp),
    'online_32bp': r_online_32bp,
    'batch_32bp':  r_batch_32bp,
    'diff_32bp':   np.abs(r_online_32bp - r_batch_32bp),
})

,track,online_1bp,batch_1bp,diff_1bp,online_32bp,batch_32bp,diff_32bp
0,0,0.917711,0.917711,1.443290e-15,0.925577,0.925577,7.771561e-16
1,1,0.915382,0.915382,8.881784e-16,0.912759,0.912759,6.106227e-15
2,2,0.912983,0.912983,5.551115e-16,0.908371,0.908371,7.771561e-15
3,3,0.920177,0.920177,7.771561e-16,0.933569,0.933569,1.665335e-15


In [7]:
np.testing.assert_allclose(r_online_1bp,  r_batch_1bp,  rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(r_online_32bp, r_batch_32bp, rtol=1e-10, atol=1e-12)
print('All assertions passed — online and batch results are numerically equivalent.')

All assertions passed — online and batch results are numerically equivalent.
